# Data Types and Casting

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Identify** when a column's stored data type does not match its intended meaning
- **Explain** why treating ordered labels as integers causes problems in analysis and modeling
- **Apply** the correct pandas casting operations to align types with meaning

> **Tip:** Select `survived` and `pclass` first. These are the two most commonly misused columns in the Titanic dataset, and understanding why their types matter will help you recognize the same pattern in other datasets.

---
## How we got here

In `02_loading_and_inspecting_data.ipynb` we ran `dtypes` and `info()` on the Titanic dataset and saw that most columns load as `int64` or `object`. Now we dig into what those types actually mean for each column, and where the stored type conflicts with the column's real meaning.

---
## Why this matters for data science

Machine learning models do math on your data. If a column containing an ordered label like passenger class (1, 2, 3) is stored as `int64`, a model will assume that 3rd class is numerically three times something, or that the difference between 1st and 2nd class equals the difference between 2nd and 3rd. That assumption is false. Getting types right before modeling is not just a housekeeping task. It prevents the model from learning the wrong relationships.

---
## Try it yourself

In [ ]:
out = Output()

COLUMN_INFO = {
    "survived": {
        "current_dtype": "int64",
        "should_be": "bool or category",
        "why": "Survived is a binary outcome (0 or 1), not a number you would add or average. Treating it as an integer implies a numeric relationship that does not exist.",
        "cast_code": "titanic['survived'] = titanic['survived'].astype(bool)",
    },
    "pclass": {
        "current_dtype": "int64",
        "should_be": "category (ordered)",
        "why": "Passenger class is an ordered label (1st, 2nd, 3rd), not a number. Treating it as an integer tells a model that 3rd class is '3 times' something, which is meaningless.",
        "cast_code": "titanic['pclass'] = pd.Categorical(titanic['pclass'], categories=[1, 2, 3], ordered=True)",
    },
    "sex": {
        "current_dtype": "object",
        "should_be": "category",
        "why": "Sex is already stored as strings, but converting to category saves memory and makes groupby operations faster.",
        "cast_code": "titanic['sex'] = titanic['sex'].astype('category')",
    },
    "age": {
        "current_dtype": "float64",
        "should_be": "float64 (already correct)",
        "why": "Age is a continuous numeric variable. float64 is the right type. Missing values (NaN) are stored correctly as float.",
        "cast_code": "# No cast needed — age is already float64",
    },
    "fare": {
        "current_dtype": "float64",
        "should_be": "float64 (already correct)",
        "why": "Fare is a continuous price value. float64 is appropriate. No transformation needed at the type level.",
        "cast_code": "# No cast needed — fare is already float64",
    },
    "embarked": {
        "current_dtype": "object",
        "should_be": "category",
        "why": "Port of embarkation (C, Q, S) is a nominal categorical variable with no inherent order. Category type is more appropriate and memory-efficient than object.",
        "cast_code": "titanic['embarked'] = titanic['embarked'].astype('category')",
    },
    "sibsp": {
        "current_dtype": "int64",
        "should_be": "int64 (already correct)",
        "why": "Number of siblings or spouses is a true integer count. int64 is the right type here.",
        "cast_code": "# No cast needed — sibsp is already int64",
    },
    "parch": {
        "current_dtype": "int64",
        "should_be": "int64 (already correct)",
        "why": "Number of parents or children is a true integer count. int64 is correct.",
        "cast_code": "# No cast needed — parch is already int64",
    },
}

col_dropdown = Dropdown(
    options=list(COLUMN_INFO.keys()),
    value="survived",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="360px"),
)

def render(col):
    info = COLUMN_INFO[col]
    html_content = (
        f'<div style="font-family: Inter, Arial, sans-serif; max-width: 600px;">'
        f'<table style="border-collapse: collapse; width: 100%; font-size: 14px;">'
        f'<tr><td style="padding: 8px 12px; background: #F1F3F5; font-weight: 600; width: 160px;">Current dtype</td>'
        f'<td style="padding: 8px 12px; color: #F76707; font-family: monospace;">{info["current_dtype"]}</td></tr>'
        f'<tr><td style="padding: 8px 12px; background: #F1F3F5; font-weight: 600;">Should be</td>'
        f'<td style="padding: 8px 12px; color: #2F9E44; font-family: monospace;">{info["should_be"]}</td></tr>'
        f'<tr><td style="padding: 8px 12px; background: #F1F3F5; font-weight: 600; vertical-align: top;">Why</td>'
        f'<td style="padding: 8px 12px; color: #212529; line-height: 1.6;">{info["why"]}</td></tr>'
        f'<tr><td style="padding: 8px 12px; background: #F1F3F5; font-weight: 600;">Cast code</td>'
        f'<td style="padding: 8px 12px; font-family: monospace; background: #212529; color: #E9ECEF; border-radius: 4px;">'
        f'{info["cast_code"]}</td></tr>'
        f'</table></div>'
    )
    with out:
        clear_output(wait=True)
        display(HTML(html_content))

def on_change(change):
    render(col_dropdown.value)

col_dropdown.observe(on_change, names="value")

display(VBox([col_dropdown, out]))
render(col_dropdown.value)

---
## What's happening?

Every column in a pandas DataFrame has a dtype that determines how pandas and scikit-learn process it. The most important types to understand are:

| dtype | What it means | When to use it |
|-------|--------------|----------------|
| `int64` | Whole number — can be added, multiplied, ordered | True counts and integers |
| `float64` | Decimal number — same arithmetic, allows NaN | Continuous measurements |
| `object` | Python string — no numeric meaning | Free text, unstructured labels |
| `category` | Finite set of values with optional ordering | Labels, groups, classes |
| `bool` | True or False | Binary outcomes |

The casting operations are:

```python
# Convert to boolean
df['survived'] = df['survived'].astype(bool)

# Convert to ordered categorical
df['pclass'] = pd.Categorical(df['pclass'], categories=[1, 2, 3], ordered=True)

# Convert to unordered categorical
df['sex'] = df['sex'].astype('category')
```

---
## Real-world example: Titanic Dataset

The chart below shows the current dtype of each Titanic column as loaded by seaborn. Orange means `int64`, blue means `float64`, green means `object`.

Notice:
- **`survived`** and **`pclass`** are both `int64`, but neither represents a true integer quantity
- **`sex`** and **`embarked`** are `object` (string), but they would be more memory-efficient and semantically clearer as `category`
- **`age`** and **`fare`** are correctly stored as `float64`

> **Discussion question:** Why would treating `pclass` as a number be misleading for a model? If 1st class is 1 and 3rd class is 3, what false arithmetic relationship does that imply?

In [ ]:
# Show current dtype classification for key Titanic columns
columns = ["survived", "pclass", "sex", "age", "fare", "sibsp", "parch", "embarked"]
dtypes = [str(titanic[c].dtype) for c in columns]

dtype_color_map = {
    "int64": PALETTE["secondary"],
    "float64": PALETTE["primary"],
    "object": PALETTE["accent"],
    "category": PALETTE["muted"],
    "bool": "#CC5DE8",
}
colors = [dtype_color_map.get(d, PALETTE["muted"]) for d in dtypes]

fig = go.Figure(
    data=[go.Bar(
        x=columns,
        y=[1] * len(columns),
        marker_color=colors,
        text=dtypes,
        textposition="inside",
        insidetextanchor="middle",
        hovertemplate="%{x}: %{text}<extra></extra>",
    )],
    layout=base_layout(
        title="Current Data Types in the Titanic Dataset",
        xaxis_title="Column",
        yaxis_title="",
    ),
)
fig.update_layout(
    yaxis=dict(showticklabels=False, showgrid=False),
    showlegend=False,
    height=280,
)
fig.show()

### Type conversion patterns across common datasets

| Column type | Stored as | Should be | Why |
|-------------|-----------|-----------|-----|
| Binary outcome (yes/no, 0/1) | int64 | bool or category | Not a number to sum or average |
| Ordered rating (1-5 stars) | int64 | Ordered category | Gaps between values may not be equal |
| Unordered label (color, city) | object | category | Enables groupby, saves memory |
| Postal code or ID | int64 | object or category | Math on IDs is meaningless |
| Date stored as string | object | datetime64 | Enables date arithmetic and resampling |

---
## Key takeaway

> **A column's data type should reflect its meaning, not just its storage format — mismatched types silently corrupt every analysis and model that uses them.**

---
*Next up: 04_missing_data.ipynb — understanding what is absent from the dataset and deciding what to do about it*